In [36]:
from CoolProp.CoolProp import PropsSI as PS
import numpy as np                            # arrays, for looping over temperatures
import matplotlib.pyplot as plt               # plots for your sensitivity figures

fl  = 'R1233zd(E)'
cpw = 4180.0
T_DC_out = 60 + 273.15 # Water temp. coming out of DC (K)
m_w = 28.7 # Water loop flow rate for DC cooling (kg/s) 
PP, SH, SC = 5.0, 5.0, 3.0 # Pinch (K), superheat (K), subcool (K)
eta_t, eta_p = 0.8, 0.75 # Efficiencies of ORC turbine and pump

# Pattern of fluid is PS(output, input1, value1, input2, value2, fluid)
# Q is vapour quality: 0 is saturated liquid, 1 is saturated vapour

def orc(T_evap, T_cond):
    P_e, P_c = PS('P','T',T_evap,'Q',1,fl), PS('P','T',T_cond,'Q',1,fl) # Gives evaporator and condenser pressures - check if this should be 0 for saturated liquid

    # 1: pump inlet (sat. liquid at T_cond, minus subcooling)
    T1 = T_cond - SC # Pump inlet temp.
    h1 = PS('H','T',T1,'P',P_c,fl);  s1 = PS('S','T',T1,'P',P_c,fl) # Enthalpy and entropy at pump inlet
    
    # 2: pump exit
    h2_ideal = PS('H','P',P_e,'S',s1,fl) # Calculates the enthalpy at pump exit if it were isentropic (pump just increases pressure from P_c to P_e and entropy stays the same)
    h2 = h1 + (h2_ideal - h1)/eta_p # Calculates the actual enthalpy at pump exit, accounting for pump efficiency (note, isentropic efficiency is used)
    
    # 3: WF liquid preheated to evaporation temperature
    h3 = PS('H','T',T_evap,'Q',0,fl)  # Enthalpy of saturated liquid at evaporator temperature
    
    # 4: turbine inlet
    h4 = PS('H','T',T_evap+SH,'P',P_e,fl);  s4 = PS('S','T',T_evap+SH,'P',P_e,fl) # Calculates the enthalpy and entropy at turbine inlet (superheated vapor)
    
    # 5: turbine outlet
    h5_ideal = PS('H','P',P_c,'S',s4,fl) # Calculates the enthalpy at turbine exit if it were isentropic (turbine just decreases pressure from P_e to P_c and entropy stays the same)
    h5   = h4 - eta_t*(h4 - h5_ideal) # Calculates the actual enthalpy at turbine exit, accounting for turbine efficiency (note, isentropic efficiency is used)

    # pinch at bubble point: water temp there = T_evap + PP
    Q_boil = m_w*cpw*(T_DC_out - (T_evap+PP)) # Heat for boiling + superheat (note the evaporator is counter-flow, so water first superheats + boils the working fluid, then preheats it)
    m_wf = Q_boil/(h4 - h3) # mass flow rate of working fluid in the ORC loop
    Q_in = m_wf*(h4 - h2) # Heat delivered to the ORC working fluid in the evaporator
    To = T_DC_out - Q_in/(m_w*cpw) # check To >= T_evap-ish and cold-end pinch

    W_t, W_p = m_wf*(h4-h5), m_wf*(h2-h1)    # Work output from turbine and work input to pump
    Q_cond = m_wf*(h5 - h1) # Heat rejected to the condenser
    return dict(m_wf=m_wf, Q_in=Q_in, To=To, W_net=W_t-W_p, Q_cond=Q_cond, OCR_efficiency=100*(W_t-W_p)/Q_in)

Tcond is not chosen, it is set by the cold sink temperature. T_cond = wet bulb + approach + range + pinch.

Wet bulb - obtained from weather data

Approach temp - choose constant 4 degrees from "Modelling the influence paper"

Range - chosen, Claude and Thermo-economic comparison of air-cooled and cooling tower recommend around 10 F = 5K range (but then should iterate to find optimum)

Condensor pinch point - 5K, from the table 1 in Thermo-economic comparison of air-cooled and cooling tower 

In [24]:
T_wb = 30 + 273.15
T_PP_condensor = 5
T_approach = 4
T_range = 5
T_cond = T_wb + T_approach + T_range + T_PP_condensor

print(orc(50+273.15, T_cond))

{'m_wf': 3.2971592737590725, 'Q_in': 636257.8388155613, 'To': 327.8463486419855, 'W_net': 8992.89731300668, 'Q_cond': 627264.9415025546, 'OCR_efficiency': 1.413404560916309}


Now calculate the cooling tower pump power, based of the "Modelling the Influence of Climate on the Performance of the Organic Rankine Cycle for Industrial Waste Heat Recovery" paper.

This assumes a pressure drop aross the cooling tower pump of 100 kPa

In [34]:
# Call function to get Q_cond
Q_cond = orc(50+273.15, T_cond)['Q_cond']

# Approximate density of water at condenser temperature
rho_w = 1000 # kg/m^3 (approximate density of water)

# Calculate volume flow rate of water in condensor loop (not mass flow rate)
V_w_ct = Q_cond/(rho_w*cpw*(T_range))

# Define pressure drop in condenser loop (Pa)
dp = 100e3

# Define pump motor and impeller efficiencies
eta_m, eta_imp = 0.9, 0.78 

# Calculate the work required to pump the water through the condenser loop
W_ctp = V_w_ct * dp / (eta_m * eta_imp)
print(f"Nominal cooling tower pump power: {W_ctp/1e3:.2f} kW")

Nominal cooling tower pump power: 4.28 kW


Now calculate the power needed for the fan that cools the water in the tower. 

Basing this off equation (20) from "Thermo-economic comparison of air-cooled and cooling tower based Organic Rankine Cycle (ORC) with R245fa and R1233zde as candidate working fluids for different geographical climate conditions", rearranged for air flow.
$$
\dot{Q}_{r e j}=\frac{c_1 \cdot m_{s i}^{c_3}}{1+c_2 \cdot\left(\frac{m_{s i}}{m_a}\right)^{c_3}}\left(T_{c w s}-T_{w b, a i r}\right)
$$

Then calculate fan power based on the fact power scales with the cube of air flow (need a constant of proportionality)

In [37]:
# Define constants for cooling tower air flow calculation (from paper)
c1, c2, c3 = 4.43, 1.12, 1.11   

def air_flow_needed(Q, m_w, dT):
    x = c1*dT/Q - m_w**(-c3)
    if x <= 0:
        raise ValueError("duty exceeds what this tower can reject at this water flow")
    return (c2/x)**(1/c3)

# calibrate at the design point (their tower: 2461.83 kW, 80.44 kg/s, 15 kW fan)
m_a_nom = air_flow_needed(Q_nom, m_w_nom, dT_nom)

# any other operating point
m_a   = air_flow_needed(Q_cond, m_w_ct, dT)
P_fan = P_fan_nom * (m_a / m_a_nom)**3

NameError: name 'Q_nom' is not defined